# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

###**Finding #4 — The Freshness Multiplier**

The paper found that older pages that were recently refreshed performed better than older pages that were not refreshed. It reported a 3.2x increase in health score and much higher impressions for the refreshed pages.

My question is: How were the refreshed and non-refreshed pages compared? For example, were the pages already different in terms of traffic, quality, or search visibility before the refresh?

The result shows a strong difference between the two groups, but this alone does not prove that the refresh caused the improvement. It is still useful as a directional finding, but more controlled testing would make the claim stronger.

Finding #10 — AI Model Performance

The paper compared OpenAI and Gemini content across different age groups. It found that neither model was always better, with each performing better in some age groups.

My question is: Were the pages being compared similar enough apart from the AI model used to produce them?

Things like topic, content age, competition, and page quality could also affect performance. So, the result is better treated as an observed difference between the groups rather than proof that one AI model is better than the other.

In [ ]:
paper_findings = {
    "Finding #4": "The Freshness Multiplier",
    "Finding #10": "AI Model Performance"
}

print("Paper findings selected for methodology audit:")
for finding, title in paper_findings.items():
    print(f"{finding}: {title}")

print("\nMethodology questions focus on:")
print("- How the comparison groups and labels were defined")
print("- Whether the validation/comparison design supports the strength of the claim")
print("- Whether observed differences can be interpreted as causal")

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model used a grouped-by-client split, meaning pages from the same client were kept on only one side of the train/test split.

For this audit, I compare that with a normal random row split. The random split is easier for the model because pages from the same client can end up in both train and test. The grouped split is more honest because the model is tested on clients it did not see during training.

I use the same Logistic Regression model, features, and F1 metric for both splits.

I also removed ctr and engagement_rate from the features because they were used directly to create `refresh_priority`. Including them would give the model information that was already used to create the target.

In [ ]:
!git clone https://github.com/martindiarua/ML_01.git
%cd ML_01/data/raw

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Load the dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Target
target = "refresh_priority"

# Features deliberately exclude CTR and engagement_rate
# because both were used to engineer refresh_priority.
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

X = df[features].copy()
y = df[target].copy()

# Logistic Regression pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# --------------------------------------------------
# BEFORE: random row-level split
# --------------------------------------------------

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model.fit(X_train_random, y_train_random)

random_pred = model.predict(X_test_random)

random_f1 = f1_score(y_test_random, random_pred)
random_accuracy = accuracy_score(y_test_random, random_pred)
random_precision = precision_score(y_test_random, random_pred)
random_recall = recall_score(y_test_random, random_pred)

# --------------------------------------------------
# AFTER: grouped-by-client split
# --------------------------------------------------

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

model.fit(X_train_grouped, y_train_grouped)

grouped_pred = model.predict(X_test_grouped)

grouped_f1 = f1_score(y_test_grouped, grouped_pred)
grouped_accuracy = accuracy_score(y_test_grouped, grouped_pred)
grouped_precision = precision_score(y_test_grouped, grouped_pred)
grouped_recall = recall_score(y_test_grouped, grouped_pred)

# --------------------------------------------------
# Comparison table
# --------------------------------------------------

comparison = pd.DataFrame({
    "split": [
        "Random row split",
        "Grouped by client"
    ],
    "accuracy": [
        random_accuracy,
        grouped_accuracy
    ],
    "precision": [
        random_precision,
        grouped_precision
    ],
    "recall": [
        random_recall,
        grouped_recall
    ],
    "f1": [
        random_f1,
        grouped_f1
    ]
})

display(comparison)

print("Random split train rows:", len(X_train_random))
print("Random split test rows:", len(X_test_random))

print("\nGrouped split train rows:", len(X_train_grouped))
print("Grouped split test rows:", len(X_test_grouped))

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("\nGrouped split train clients:", len(train_clients))
print("Grouped split test clients:", len(test_clients))
print("Shared clients:", len(train_clients.intersection(test_clients)))

### Validation finding

The random split gave an F1 score of **[RANDOM F1]**, while the grouped-by-client split gave an F1 score of **[GROUPED F1]**.

The grouped split is the more honest test for this dataset because there are no clients shared between the training and test sets.

In Week 5, the Logistic Regression model had an F1 score of about **0.922**, compared with **0.803** for the baseline. This shows that the model performed better than the baseline on the tested data.

However, the result should still be treated as evidence from this dataset and split, not as a guarantee that the model will perform the same way on every future client.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The target `refresh_priority` was created using ctr and engagement_rate.

Because of this, I removed both fields from the model features. Using them would create leakage because the model would have access to the same information that was used to create the target.

I also checked that leak_feature from Week 3 and the future-window fields were not included in the final feature set.

The final model therefore does not use the known fields that could directly leak the target or future information.

In [ ]:
# Final feature list
print("Number of final features:", len(features))
print("\nFinal features:")
print(features)

# Direct target-construction fields
target_construction_fields = [
    "ctr",
    "engagement_rate"
]

print("\nTarget-construction leakage check:")

for column in target_construction_fields:
    print(
        f"{column}:",
        "USED AS FEATURE" if column in features else "EXCLUDED"
    )

# Known leakage feature from Week 3
known_leakage_features = [
    "leak_feature",
    "refresh_priority"
]

print("\nKnown leakage fields:")

for column in known_leakage_features:
    print(
        f"{column}:",
        "USED AS FEATURE" if column in features else "EXCLUDED"
    )

# Future-window fields
future_fields = [
    "impressions_future",
    "clicks_future",
    "sessions_future"
]

print("\nFuture-window leakage check:")

for column in future_fields:
    print(
        f"{column}:",
        "USED AS FEATURE" if column in features else "EXCLUDED"
    )

# Explicit final check
forbidden_features = (
    target_construction_fields
    + known_leakage_features
    + future_fields
)

leakage_found = [
    column for column in forbidden_features
    if column in features
]

print("\nFinal leakage check:")

if len(leakage_found) == 0:
    print("PASS — no identified target-construction, known leakage, or future-window fields are used.")
else:
    print("REVIEW REQUIRED — forbidden features found:", leakage_found)

### Leakage finding

The leakage check passed.

`ctr` and `engagement_rate` were excluded because they were used to create `refresh_priority`. `leak_feature` and the future-window fields were also excluded.

This does not mean that every possible form of leakage has been ruled out, but the specific leakage issues identified during the earlier weeks are not present in the final feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My strongest Week-5 claim was that **the Logistic Regression model performed better than the baseline**.

The measured results were:

- Week-4 baseline F1: **0.803**
- Week-5 Logistic Regression F1: **0.922**
### **Original claim**

Logistic Regression significantly outperforms the baseline and provides a much better way to identify pages that should be refreshed.

### **Safer claim**

On the test data used in this project, Logistic Regression achieved a higher F1 score of **0.922** than the Week-4 baseline score of **0.803**. This shows that the model was better at predicting the engineered `refresh_priority` label on the tested data. It does not prove that the model can identify pages that will definitely benefit from a refresh.

The reason for changing the claim is simple: `refresh_priority` was created from a rule using `ctr` and `engagement_rate`. So the model is predicting that label, not directly predicting whether a real page refresh will improve future performance.

## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.